In [ ]:
# Install required libraries silently
!pip install tiktoken huggingface_hub accelerate -q

In [ ]:
# Clone your GitHub repository (Replace the URL with your actual GitHub repo link)
!git clone https://github.com/krisnaganesha1609/Conditioned-Pantun-Transformer.git

# Change the working directory to the cloned repository
%cd Conditioned-Pantun-Transformer

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Make sure you have added your Hugging Face Token in Kaggle Secrets named "HF_TOKEN"
# Go to Add-ons -> Secrets in the Kaggle top menu to set this up.
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Successfully logged into Hugging Face!")
except Exception as e:
    print(f"Error authenticating: {e}")
    print("Please ensure 'HF_TOKEN' is set in Kaggle Secrets.")

In [ ]:
# Verify the pipeline works end-to-end on the Kaggle GPUs before committing to a long run
!python 3.TRAINING/train.py --smoke_test# Execute the full training run. 

In [ ]:
# Execute the full training run. 
# Using batch_size 32 and grad_accum_steps 2 effectively gives a batch size of 64 per step.
# The DataParallel wrapper inside train.py will automatically utilize both T4 GPUs.
!python 3.TRAINING/train.py \
    --batch_size 32 \
    --grad_accum_steps 2 \
    --warmup_steps 300 \
    --max_steps 5000 \
    --eval_interval 250 \
    --eval_iters 100 \
    --lr 6e-4 \
    --min_lr 6e-5

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
repo_id = "igkganes/conditioned-pantun-transformer"

print(f"Preparing to push to {repo_id}...")

# Create repo if it doesn't exist
api.create_repo(repo_id=repo_id, exist_ok=True, repo_type="model")

# Upload the checkpoints folder only
api.upload_folder(
    folder_path="3.TRAINING/checkpoints",
    repo_id=repo_id,
    repo_type="model",
    path_in_repo="checkpoints",
    commit_message="Upload Kaggle T4x2 training checkpoints"
)

print(f"Successfully pushed checkpoints to https://huggingface.co/{repo_id}")